In [ ]:
# ===== PURE PYTORCH CCNET INFERENCE (NO MMSEG / MMCV) =====
# 1) Sua cac path ben duoi
# 2) Chay cell
# 3) Neu load_state_dict bao missing/unexpected keys, gui log do

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================================================
# USER SETTINGS
# =========================================================
CKPT_PATH = r"./CCNet/iter_80000.pth"          # <- sua neu can
CLASS_MAPPING_PATH = r"./datasets/foodseg103-full/class_mapping.json"    # <- sua neu can
DATA_ROOT = r"./datasets/foodseg103-full"
WORK_DIR = r"./outputs/ccnet_eval"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# test resize: chon 1 trong 2
# file config upload: (1024,1024)
# file meta khac:      (2048,1024)
TEST_SCALE = (1024, 1024)   # (W, H)
# TEST_SCALE = (2048, 1024) # neu muon bam file meta kia

MAX_ITEMS = None   # dat None neu muon chay full
os.makedirs(WORK_DIR, exist_ok=True)

assert os.path.exists(CKPT_PATH), CKPT_PATH
assert os.path.exists(CLASS_MAPPING_PATH), CLASS_MAPPING_PATH
assert os.path.exists(DATA_ROOT), DATA_ROOT

TEST_IMG_DIR = os.path.join(DATA_ROOT, "test", "img")
TEST_MASK_DIR = os.path.join(DATA_ROOT, "test", "mask")
TEST_ANN_DIR = os.path.join(DATA_ROOT, "test", "ann")

assert os.path.exists(TEST_IMG_DIR), TEST_IMG_DIR
assert os.path.exists(TEST_MASK_DIR), TEST_MASK_DIR
assert os.path.exists(TEST_ANN_DIR), TEST_ANN_DIR

# =========================================================
# CLASS MAPPING
# =========================================================
with open(CLASS_MAPPING_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

ID_TO_CLASS = {int(k): v for k, v in class_mapping["id_to_class"].items()}
BACKGROUND_ID = int(class_mapping["background_id"])
NUM_INGREDIENT_CLASSES = int(class_mapping["num_ingredient_classes"])
NUM_CLASSES = NUM_INGREDIENT_CLASSES + 1

def get_class_name(class_id):
    class_id = int(class_id)
    if class_id == BACKGROUND_ID:
        return "background"
    return ID_TO_CLASS.get(class_id, f"class_{class_id}")

print("DEVICE:", DEVICE)
print("NUM_CLASSES:", NUM_CLASSES)
print("BACKGROUND_ID:", BACKGROUND_ID)

# =========================================================
# BASIC BLOCKS
# =========================================================
def conv3x3(in_ch, out_ch, stride=1, dilation=1):
    return nn.Conv2d(
        in_ch, out_ch, kernel_size=3, stride=stride,
        padding=dilation, dilation=dilation, bias=False
    )

def conv1x1(in_ch, out_ch, stride=1):
    return nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False)

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_ch, out_ch, kernel_size=kernel_size, stride=stride,
            padding=padding, dilation=dilation, bias=False
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, dilation=1, downsample=None):
        super().__init__()
        self.conv1 = conv1x1(inplanes, planes, stride=1)
        self.bn1 = nn.BatchNorm2d(planes)

        self.conv2 = conv3x3(planes, planes, stride=stride, dilation=dilation)
        self.bn2 = nn.BatchNorm2d(planes)

        self.conv3 = conv1x1(planes, planes * self.expansion, stride=1)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)
        return out

class ResNetV1c(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64

        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),  # 0
            nn.BatchNorm2d(32),                                                 # 1
            nn.ReLU(inplace=True),                                              # 2
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False), # 3
            nn.BatchNorm2d(32),                                                 # 4
            nn.ReLU(inplace=True),                                              # 5
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False), # 6
            nn.BatchNorm2d(64),                                                 # 7
            nn.ReLU(inplace=True),                                              # 8
        )
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 3, stride=1, dilation=1)
        self.layer2 = self._make_layer(128, 4, stride=2, dilation=1)
        self.layer3 = self._make_layer(256, 6, stride=1, dilation=2)
        self.layer4 = self._make_layer(512, 3, stride=1, dilation=4)

    def _make_layer(self, planes, blocks, stride=1, dilation=1):
        downsample = None
        outplanes = planes * Bottleneck.expansion

        if stride != 1 or self.inplanes != outplanes:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, outplanes, stride=stride),
                nn.BatchNorm2d(outplanes)
            )

        layers = [Bottleneck(self.inplanes, planes, stride=stride, dilation=dilation, downsample=downsample)]
        self.inplanes = outplanes
        for _ in range(1, blocks):
            layers.append(Bottleneck(self.inplanes, planes, stride=1, dilation=dilation, downsample=None))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.maxpool(x)

        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return [c1, c2, c3, c4]

# =========================================================
# CRISS-CROSS ATTENTION
# from repo speedinghzl/CCNet cc_attention/functions.py
# adjusted to run on cpu/cuda without hardcoded .cuda()
# =========================================================
def INF(B, H, W, device):
    return -torch.diag(torch.tensor(float("inf"), device=device).repeat(H), 0).unsqueeze(0).repeat(B * W, 1, 1)

class CrissCrossAttention(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.query_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.key_conv = nn.Conv2d(in_dim, in_dim // 8, kernel_size=1)
        self.value_conv = nn.Conv2d(in_dim, in_dim, kernel_size=1)
        self.softmax = nn.Softmax(dim=3)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        m_batchsize, _, height, width = x.size()

        proj_query = self.query_conv(x)
        proj_query_H = proj_query.permute(0, 3, 1, 2).contiguous().view(m_batchsize * width, -1, height).permute(0, 2, 1)
        proj_query_W = proj_query.permute(0, 2, 1, 3).contiguous().view(m_batchsize * height, -1, width).permute(0, 2, 1)

        proj_key = self.key_conv(x)
        proj_key_H = proj_key.permute(0, 3, 1, 2).contiguous().view(m_batchsize * width, -1, height)
        proj_key_W = proj_key.permute(0, 2, 1, 3).contiguous().view(m_batchsize * height, -1, width)

        proj_value = self.value_conv(x)
        proj_value_H = proj_value.permute(0, 3, 1, 2).contiguous().view(m_batchsize * width, -1, height)
        proj_value_W = proj_value.permute(0, 2, 1, 3).contiguous().view(m_batchsize * height, -1, width)

        energy_H = (torch.bmm(proj_query_H, proj_key_H) + INF(m_batchsize, height, width, x.device)).view(
            m_batchsize, width, height, height
        ).permute(0, 2, 1, 3)

        energy_W = torch.bmm(proj_query_W, proj_key_W).view(m_batchsize, height, width, width)

        concate = self.softmax(torch.cat([energy_H, energy_W], 3))
        att_H = concate[:, :, :, 0:height].permute(0, 2, 1, 3).contiguous().view(m_batchsize * width, height, height)
        att_W = concate[:, :, :, height:height + width].contiguous().view(m_batchsize * height, width, width)

        out_H = torch.bmm(proj_value_H, att_H.permute(0, 2, 1)).view(m_batchsize, width, -1, height).permute(0, 2, 3, 1)
        out_W = torch.bmm(proj_value_W, att_W.permute(0, 2, 1)).view(m_batchsize, height, -1, width).permute(0, 2, 1, 3)

        return self.gamma * (out_H + out_W) + x

# =========================================================
# HEADS
# =========================================================
class BaseDecodeHeadLite(nn.Module):
    def __init__(self, in_channels, channels, num_classes, in_index, dropout_ratio=0.1):
        super().__init__()
        self.in_channels = in_channels
        self.channels = channels
        self.num_classes = num_classes
        self.in_index = in_index
        self.dropout_ratio = dropout_ratio

        self.dropout = nn.Dropout2d(dropout_ratio) if dropout_ratio > 0 else nn.Identity()
        self.conv_seg = nn.Conv2d(channels, num_classes, kernel_size=1)

    def _transform_inputs(self, inputs):
        return inputs[self.in_index]

    def cls_seg(self, feat):
        feat = self.dropout(feat)
        return self.conv_seg(feat)

class FCNHeadLite(BaseDecodeHeadLite):
    def __init__(self, in_channels, channels, num_classes, in_index,
                 num_convs=1, concat_input=False, dropout_ratio=0.1):
        super().__init__(in_channels, channels, num_classes, in_index, dropout_ratio)
        self.num_convs = num_convs
        self.concat_input = concat_input

        convs = []
        if num_convs == 0:
            assert in_channels == channels
            self.convs = nn.Identity()
        else:
            convs.append(ConvBNReLU(in_channels, channels, kernel_size=3, padding=1))
            for _ in range(num_convs - 1):
                convs.append(ConvBNReLU(channels, channels, kernel_size=3, padding=1))
            self.convs = nn.Sequential(*convs)

        if concat_input:
            self.conv_cat = ConvBNReLU(in_channels + channels, channels, kernel_size=3, padding=1)

    def forward(self, inputs):
        x = self._transform_inputs(inputs)
        output = self.convs(x)
        if self.concat_input:
            output = self.conv_cat(torch.cat([x, output], dim=1))
        output = self.cls_seg(output)
        return output

class CCHeadLite(FCNHeadLite):
    # based on mmseg CCHead: num_convs=2, concat_input=True by FCNHead default
    def __init__(self, in_channels, channels, num_classes, in_index,
                 recurrence=2, dropout_ratio=0.1, concat_input=True):
        super().__init__(
            in_channels=in_channels,
            channels=channels,
            num_classes=num_classes,
            in_index=in_index,
            num_convs=2,
            concat_input=concat_input,
            dropout_ratio=dropout_ratio
        )
        self.recurrence = recurrence
        self.cca = CrissCrossAttention(channels)

    def forward(self, inputs):
        x = self._transform_inputs(inputs)
        output = self.convs[0](x)
        for _ in range(self.recurrence):
            output = self.cca(output)
        output = self.convs[1](output)
        if self.concat_input:
            output = self.conv_cat(torch.cat([x, output], dim=1))
        output = self.cls_seg(output)
        return output

# =========================================================
# SEGMENTOR
# =========================================================
class EncoderDecoderLite(nn.Module):
    def __init__(self, num_classes=104):
        super().__init__()
        self.backbone = ResNetV1c()
        self.decode_head = CCHeadLite(
            in_channels=2048,
            in_index=3,
            channels=512,
            recurrence=2,
            dropout_ratio=0.1,
            num_classes=num_classes,
            concat_input=True
        )
        self.auxiliary_head = FCNHeadLite(
            in_channels=1024,
            in_index=2,
            channels=256,
            num_convs=1,
            concat_input=False,
            dropout_ratio=0.1,
            num_classes=num_classes
        )

    def extract_feat(self, x):
        return self.backbone(x)

    def encode_decode(self, x):
        feats = self.extract_feat(x)
        out = self.decode_head(feats)
        out = F.interpolate(out, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return out

    def forward(self, x):
        return self.encode_decode(x)

# =========================================================
# CHECKPOINT LOAD
# =========================================================
def load_checkpoint_state(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    if "state_dict" in ckpt:
        state = ckpt["state_dict"]
    else:
        state = ckpt
    return ckpt, state

def strip_prefix_if_needed(state):
    # de-handling few common wrappers
    new_state = {}
    for k, v in state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module."):]
        new_state[nk] = v
    return new_state

def convert_syncbn_to_bn_in_state_dict(state):
    # SyncBN vs BN parameter names mostly same for weight/bias/running stats
    # so usually no change needed
    return state

def print_state_summary(state, limit=40):
    print("num tensors:", len(state))
    for i, (k, v) in enumerate(state.items()):
        if i >= limit:
            break
        print(k, tuple(v.shape))

ckpt_raw, state = load_checkpoint_state(CKPT_PATH)
state = strip_prefix_if_needed(state)
state = convert_syncbn_to_bn_in_state_dict(state)

print("checkpoint keys:", list(ckpt_raw.keys())[:10])
print_state_summary(state, limit=20)

model = EncoderDecoderLite(num_classes=NUM_CLASSES)

missing, unexpected = model.load_state_dict(state, strict=False)
print("\nmissing keys:", len(missing))
print("unexpected keys:", len(unexpected))

if len(missing) > 0:
    print("\nFirst missing keys:")
    for x in missing[:30]:
        print("  ", x)

if len(unexpected) > 0:
    print("\nFirst unexpected keys:")
    for x in unexpected[:30]:
        print("  ", x)

model.to(DEVICE).eval()

# =========================================================
# DATA / PREPROCESS
# =========================================================
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
STD  = np.array([58.395, 57.12, 57.375], dtype=np.float32)

def list_images(img_dir):
    out = []
    for name in sorted(os.listdir(img_dir)):
        p = os.path.join(img_dir, name)
        if os.path.isfile(p) and os.path.splitext(name)[1].lower() in IMG_EXTS:
            out.append(p)
    return out

def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def load_mask(path):
    return np.array(Image.open(path), dtype=np.int64)

def load_ann(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resize_keep_ratio(img_np, target_wh):
    target_w, target_h = target_wh
    h, w = img_np.shape[:2]
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    img = Image.fromarray(img_np)
    img = img.resize((new_w, new_h), Image.BILINEAR)
    return np.array(img), (new_h, new_w)

def normalize_to_tensor(img_np):
    x = img_np.astype(np.float32)
    x = (x - MEAN) / STD
    x = x.transpose(2, 0, 1)
    x = torch.from_numpy(x).float().unsqueeze(0)
    return x

@torch.no_grad()
def infer_one_image(model, img_path, test_scale=(1024, 1024)):
    img_np = load_rgb(img_path)
    ori_h, ori_w = img_np.shape[:2]

    resized, _ = resize_keep_ratio(img_np, test_scale)
    x = normalize_to_tensor(resized).to(DEVICE)

    logits = model(x)
    pred_small = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.int64)

    pred = Image.fromarray(pred_small.astype(np.uint8))
    pred = pred.resize((ori_w, ori_h), Image.NEAREST)
    pred = np.array(pred, dtype=np.int64)

    return img_np, pred

# =========================================================
# VIS
# =========================================================
def make_palette(n, seed=123):
    rng = np.random.default_rng(seed)
    palette = rng.integers(0, 255, size=(n, 3), dtype=np.uint8)
    palette[BACKGROUND_ID] = np.array([0, 0, 0], dtype=np.uint8)
    return palette

PALETTE = make_palette(NUM_CLASSES)

def colorize_mask(mask_np):
    x = np.clip(mask_np, 0, NUM_CLASSES - 1)
    return PALETTE[x]

def overlay_mask(img_np, mask_np, alpha=0.45):
    color = colorize_mask(mask_np)
    fg = (mask_np != BACKGROUND_ID)[..., None]
    over = np.where(fg, (1 - alpha) * img_np + alpha * color, img_np)
    return np.clip(over, 0, 255).astype(np.uint8)

def color_error_map(pred, gt):
    out = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)
    out[(pred == gt)] = np.array([0, 180, 0], dtype=np.uint8)
    out[(pred != gt)] = np.array([220, 30, 30], dtype=np.uint8)
    return out

# =========================================================
# METRICS
# =========================================================
def safe_div(a, b):
    return float(a) / float(b) if b != 0 else 0.0

def compute_iou_binary(pred_bin, gt_bin):
    inter = np.logical_and(pred_bin, gt_bin).sum()
    union = np.logical_or(pred_bin, gt_bin).sum()
    return safe_div(inter, union)

def compute_confusion_matrix(pred, gt, num_classes):
    pred = pred.astype(np.int64)
    gt = gt.astype(np.int64)
    inds = num_classes * gt.reshape(-1) + pred.reshape(-1)
    hist = np.bincount(inds, minlength=num_classes**2).reshape(num_classes, num_classes)
    return hist

def compute_scores_from_hist(hist):
    hist = hist.astype(np.float64)
    diag = np.diag(hist)
    aAcc = safe_div(diag.sum(), hist.sum())
    acc_cls = diag / np.maximum(hist.sum(axis=1), 1.0)
    iu = diag / np.maximum(hist.sum(axis=1) + hist.sum(axis=0) - diag, 1.0)
    return {
        "aAcc": float(aAcc),
        "mAcc": float(np.nanmean(acc_cls)),
        "mIoU": float(np.nanmean(iu)),
    }

# =========================================================
# QUICK SAMPLE
# =========================================================
test_img_files = list_images(TEST_IMG_DIR)
if MAX_ITEMS is not None:
    test_img_files = test_img_files[:MAX_ITEMS]

print("\nnum test images:", len(test_img_files))

TEST_STEM = "00000048.jpg"
sample_img_path = os.path.join(TEST_IMG_DIR, TEST_STEM)
sample_mask_path = os.path.join(TEST_MASK_DIR, TEST_STEM + ".png")

if os.path.exists(sample_img_path) and os.path.exists(sample_mask_path):
    img_np, pred = infer_one_image(model, sample_img_path, TEST_SCALE)
    gt = load_mask(sample_mask_path)

    plt.figure(figsize=(20, 5))
    plt.subplot(1, 4, 1); plt.imshow(img_np); plt.title("Image"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(overlay_mask(img_np, gt)); plt.title("GT"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(overlay_mask(img_np, pred)); plt.title("Pred"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(color_error_map(pred, gt)); plt.title("Correct/Wrong"); plt.axis("off")
    plt.tight_layout()
    plt.show()

# =========================================================
# RUN TEST
# =========================================================
rows = []
global_hist = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

for i, img_path in enumerate(test_img_files):
    stem = os.path.basename(img_path)
    stem_noext = os.path.splitext(stem)[0]

    # dataset nay dang theo pattern:
    # image: 00000048.jpg
    # mask : 00000048.jpg.png
    # ann  : 00000048.jpg.json
    mask_path = os.path.join(TEST_MASK_DIR, stem_noext + ".png")  # 00000048.png
    ann_path = os.path.join(TEST_ANN_DIR, stem + ".json")         # 00000048.jpg.json   

    if not os.path.exists(mask_path):
        print("missing mask:", mask_path)
        continue
    if not os.path.exists(ann_path):
        print("missing ann :", ann_path)
        continue

    img_np, pred = infer_one_image(model, img_path, TEST_SCALE)
    gt = load_mask(mask_path)

    hist = compute_confusion_matrix(pred, gt, NUM_CLASSES)
    global_hist += hist

    pixel_acc = safe_div((pred == gt).sum(), gt.size)

    present_classes = np.unique(gt)
    ious_present = []
    for c in present_classes:
        ious_present.append(compute_iou_binary(pred == c, gt == c))
    miou_present = float(np.mean(ious_present)) if len(ious_present) > 0 else 0.0

    rows.append({
    "stem": stem,
    "img_path": img_path,
    "mask_path": mask_path,
    "ann_path": ann_path,
    "pixel_acc": pixel_acc,
    "mIoU_present": miou_present,
    "num_present_classes": int(len(present_classes)),
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(test_img_files)}")
df_test = pd.DataFrame(rows)
scores = compute_scores_from_hist(global_hist)

# =========================================================
# ANALYSIS TABLES
# =========================================================
rows_class = []
for c in range(NUM_CLASSES):
    gt_pixels = int(global_hist[c, :].sum())
    pred_pixels = int(global_hist[:, c].sum())
    tp = int(global_hist[c, c])

    union = gt_pixels + pred_pixels - tp
    iou = safe_div(tp, union)
    acc = safe_div(tp, gt_pixels)

    rows_class.append({
        "class_id": c,
        "class_name": get_class_name(c),
        "gt_pixels": gt_pixels,
        "pred_pixels": pred_pixels,
        "tp_pixels": tp,
        "IoU": iou,
        "Acc": acc,
    })

df_class = pd.DataFrame(rows_class).sort_values("IoU", ascending=True).reset_index(drop=True)

pairs = []
for gt_c in range(NUM_CLASSES):
    for pred_c in range(NUM_CLASSES):
        if gt_c == pred_c:
            continue
        if gt_c == BACKGROUND_ID or pred_c == BACKGROUND_ID:
            continue

        cnt = int(global_hist[gt_c, pred_c])
        if cnt > 0:
            pairs.append({
                "gt_id": gt_c,
                "gt_name": get_class_name(gt_c),
                "pred_id": pred_c,
                "pred_name": get_class_name(pred_c),
                "count": cnt
            })

if len(pairs) > 0:
    df_pairs = pd.DataFrame(pairs).sort_values("count", ascending=False).reset_index(drop=True)
else:
    df_pairs = pd.DataFrame(columns=["gt_id", "gt_name", "pred_id", "pred_name", "count"])

# =========================================================
# HELPER: SHOW WORST CASES
# =========================================================
def show_worst_cases(df_test, k=8):
    if len(df_test) == 0:
        print("df_test rong, khong co case de hien thi.")
        return

    worst = df_test.sort_values("mIoU_present").head(k)

    plt.figure(figsize=(18, 4 * len(worst)))
    for i, (_, row) in enumerate(worst.iterrows()):
        img_np = load_rgb(row["img_path"])
        gt = load_mask(row["mask_path"])
        _, pred = infer_one_image(model, row["img_path"], TEST_SCALE)

        plt.subplot(len(worst), 4, i * 4 + 1)
        plt.imshow(img_np)
        plt.title(f"Image\n{row['stem']}")
        plt.axis("off")

        plt.subplot(len(worst), 4, i * 4 + 2)
        plt.imshow(overlay_mask(img_np, gt))
        plt.title("GT")
        plt.axis("off")

        plt.subplot(len(worst), 4, i * 4 + 3)
        plt.imshow(overlay_mask(img_np, pred))
        plt.title("Pred")
        plt.axis("off")

        plt.subplot(len(worst), 4, i * 4 + 4)
        plt.imshow(color_error_map(pred, gt))
        plt.title("Correct / Wrong")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# =========================================================
# ERROR ANALYSIS HEURISTICS
# =========================================================
def analyze_error_patterns(df_test, df_class, df_pairs, background_id=103):
    notes = []

    # 1) rare-class fail
    df_fg = df_class[df_class["class_id"] != background_id].copy()
    rare = df_fg[df_fg["gt_pixels"] > 0].sort_values("gt_pixels").head(15)
    rare_bad = rare[rare["IoU"] < 0.05]
    if len(rare_bad) > 0:
        notes.append({
            "pattern": "rare-class fail",
            "evidence": f"{len(rare_bad)} lop hiem co IoU < 0.05",
            "direction": "class-balanced sampling / weighted CE-focal-dice / repeat factor sampler"
        })

    # 2) texture confusion
    if len(df_pairs) > 0:
        top_pairs = df_pairs.head(10)
        notes.append({
            "pattern": "texture or semantic confusion",
            "evidence": "top confusion pairs xuat hien nhieu cap nham lan lap lai",
            "direction": "texture-aware local branch / local contrast / hard example mining theo class-pair"
        })

    # 3) boundary / local detail fail
    if len(df_test) > 0:
        if df_test["pixel_acc"].mean() - df_test["mIoU_present"].mean() > 0.20:
            notes.append({
                "pattern": "boundary or local-detail fail",
                "evidence": "pixel_acc cao hon mIoU_present kha nhieu",
                "direction": "boundary loss / edge supervision / refine high-res features"
            })

    # 4) hard images / multi-object scenes
    if len(df_test) > 0:
        hard_many = df_test.sort_values("mIoU_present").head(20)
        if hard_many["num_present_classes"].mean() >= 4:
            notes.append({
                "pattern": "co-occurrence / multi-ingredient fail",
                "evidence": "anh loi nang thuong co nhieu class cung xuat hien",
                "direction": "context + local fusion / relation-aware decoder / crop strategy giu nhieu ingredient"
            })

    # 5) small-object suspicion
    if len(df_test) > 0 and len(df_class) > 0:
        very_low_iou = df_fg[(df_fg["gt_pixels"] > 0) & (df_fg["IoU"] < 0.03)]
        if len(very_low_iou) > 0:
            notes.append({
                "pattern": "small-object or thin-structure fail",
                "evidence": f"{len(very_low_iou)} lop co IoU rat thap",
                "direction": "high-resolution branch / better feature fusion / larger test scale"
            })

    return pd.DataFrame(notes)

df_error_analysis = analyze_error_patterns(df_test, df_class, df_pairs, background_id=BACKGROUND_ID)

# =========================================================
# SUMMARY
# =========================================================
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("Num test images:", len(df_test))

if len(df_test) == 0:
    print("df_test rong, kiem tra lai mapping image/mask/ann")
else:
    print("Mean image-wise pixel_acc   :", df_test["pixel_acc"].mean())
    print("Mean image-wise mIoU_present:", df_test["mIoU_present"].mean())
    print("Dataset aAcc:", scores["aAcc"])
    print("Dataset mAcc:", scores["mAcc"])
    print("Dataset mIoU:", scores["mIoU"])

    print("\nWorst 10 classes:")
    print(df_class.head(10)[["class_id", "class_name", "IoU", "Acc", "gt_pixels"]].to_string(index=False))

    if len(df_pairs) > 0:
        print("\nTop 10 confusion pairs:")
        print(df_pairs.head(10)[["gt_name", "pred_name", "count"]].to_string(index=False))
    else:
        print("\nTop 10 confusion pairs: none")

    print("\nWorst 10 images:")
    print(df_test.sort_values("mIoU_present").head(10).to_string(index=False))

    if len(df_error_analysis) > 0:
        print("\nSuggested error patterns -> improvement directions:")
        print(df_error_analysis.to_string(index=False))
    else:
        print("\nSuggested error patterns -> improvement directions: no strong heuristic pattern found")

# =========================================================
# PLOTS
# =========================================================
if len(df_test) > 0:
    # worst classes
    top_k = min(20, len(df_class))
    df_plot = df_class.head(top_k)

    plt.figure(figsize=(10, max(6, top_k * 0.35)))
    plt.barh(df_plot["class_name"], df_plot["IoU"])
    plt.xlabel("IoU")
    plt.title(f"Worst {top_k} classes by IoU")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    # top confusion pairs
    if len(df_pairs) > 0:
        df_plot = df_pairs.head(20)
        labels = [f"{r.gt_name} -> {r.pred_name}" for _, r in df_plot.iterrows()]
        vals = df_plot["count"].values

        plt.figure(figsize=(10, max(6, len(labels) * 0.35)))
        plt.barh(labels, vals)
        plt.xlabel("Misclassified pixels")
        plt.title("Top confusing class pairs")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

    # image-level metric distribution
    plt.figure(figsize=(10, 4))
    plt.hist(df_test["mIoU_present"], bins=30)
    plt.xlabel("Image-wise mIoU_present")
    plt.ylabel("Count")
    plt.title("Distribution of image-wise mIoU_present")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.hist(df_test["pixel_acc"], bins=30)
    plt.xlabel("Image-wise pixel_acc")
    plt.ylabel("Count")
    plt.title("Distribution of image-wise pixel_acc")
    plt.tight_layout()
    plt.show()

    # hardest cases
    show_worst_cases(df_test, k=min(8, len(df_test)))

# =========================================================
# SAVE CSV
# =========================================================
df_test.to_csv(os.path.join(WORK_DIR, "pure_ccnet_test_results.csv"), index=False)
df_class.to_csv(os.path.join(WORK_DIR, "pure_ccnet_per_class_results.csv"), index=False)
df_pairs.to_csv(os.path.join(WORK_DIR, "pure_ccnet_top_confusion_pairs.csv"), index=False)
df_error_analysis.to_csv(os.path.join(WORK_DIR, "pure_ccnet_error_analysis.csv"), index=False)

print("\nSaved:")
print(os.path.join(WORK_DIR, "pure_ccnet_test_results.csv"))
print(os.path.join(WORK_DIR, "pure_ccnet_per_class_results.csv"))
print(os.path.join(WORK_DIR, "pure_ccnet_top_confusion_pairs.csv"))
print(os.path.join(WORK_DIR, "pure_ccnet_error_analysis.csv"))